In [ ]:
#Installing some dependencies
!pip install -q transformers accelerate anthropic openai

In [ ]:
#Imports

import os
import torch
from google.colab import userdata
from transformers import pipeline
import anthropic
from openai import OpenAI
import gradio as gr

In [ ]:
# Setting some environment variables
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

device = "cude" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

printf(f"Using device:{device}")


In [ ]:
# Load Whisper (HuggingFace Pipeline)

transcriber = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True
)
print("whisper loaded successfully...")

In [ ]:
# Initialize Claude and OpenAI clients

claude = anthropic.Anthropic()
openai_client = OpenAI()
print("Claude and OpenAI clients initialized successfully...")

In [ ]:
# Analysis system prompt + Function

SYSTEM_PROMPT = """You are an expert music critic and literary analyst. Given song lyrics, provide a deep, thoughtful analysis.

Structure your response in Markdown with these sections:

## Themes & Subject Matter
What is this song really about? Core ideas explored.

## Emotional Tone
The dominant emotions and atmosphere conveyed.

## Literary Devices
Metaphors, symbolism, imagery, wordplay used in the lyrics.

## Hidden Meanings
Subtext, deeper interpretations, possible biographical or cultural context.

## Mood & Visual Description
A vivid sensory description that could be translated into artwork.

At the very end, on a new line write exactly:
COVER_PROMPT: [a detailed visual prompt for AI image generation — focus on mood, colors, art style, key imagery, lighting. No text or letters in the image.]

Be insightful, articulate, and avoid clichés."""

def analyze_lyrics(lyrics):
    response = claude.messages.create(
        model="claude-sonnet-4-5",
        max_tokens = 2000,
        system= SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": f"Analyze these song lyrics:\n\n{lyrics}"}
        ]
    )
    return response.content[0].text

In [ ]:
# Cover Art/Image Generation

def generate_cover_art(prompt):
  response = openai_client.images.generate(
      model="dall-e-3",
      prompt=f"Album cover art:{prompt}. Artistic, evocative, professional album cover design. No text, no words, no letters.",
      size="1024x1024",
      quality="standard",
      n=1,
  )
  return response.data[0].url

In [ ]:
# Main Pipeline

def process_song(audio_file):
  if audio_file is None:
    return "⚠️ Please upload an audio file.", "", None

  yield "🎙️ Transcribing lyrics with Whisper...", "", None
  transcription = transcriber(audio_file)

  lyrics = transcription["text"]

  yield "🔍 Analyzing lyrics...", "", None
  analysis = analyze_lyrics(lyrics)

  if "COVER_PROMPT" in analysis:
    cover_prompt = analysis.split("COVER_PROMPT:")[-1].strip()
    display_analysis = analysis.split("COVER_PROMPT:")[0].strip()
  else:
        cover_prompt = "abstract artistic visualization of music and emotion"
        display_analysis = analysis

  yield lyrics, display_analysis, None
  image_url = generate_cover_art(cover_prompt)

  yield lyrics, display_analysis, image_url
